#Camada de Streaming

**Este notebook tem como objetivo implementar um pipeline de processamento incremental utilizando Apache Spark Structured Streaming e Delta Lake. O pipeline terá como base a chegada incremental de arquivos contendo dados de viagens do conjunto de dados NYC Taxi.**

**1 - Configuração do ambiente**

In [0]:
from pyspark.sql.functions import *

In [0]:
print(f"Versão do Spark: {spark.version}")

spark.sql("SELECT current_catalog()").show()

spark.sql("SHOW SCHEMAS IN nyc_taxi_data").show()

Versão do Spark: 4.1.0
+-----------------+
|current_catalog()|
+-----------------+
|        workspace|
+-----------------+

+------------------+
|      databaseName|
+------------------+
|            bronze|
|           default|
|              gold|
|information_schema|
|               raw|
|            silver|
+------------------+



**2 - Definição da fonte de dados**

Para a implementação do streaming serão utilizados arquivos históricos do NYC Yellow Taxi disponibilizados no ambiente Databricks.

Os arquivos mensais serão utilizados para simular a chegada incremental de novos dados ao pipeline.

In [0]:
path_fonte = "/databricks-datasets/nyctaxi/tripdata/yellow/"

display(dbutils.fs.ls(path_fonte))

path,name,size,modificationTime
dbfs:/databricks-datasets/nyctaxi/tripdata/yellow/yellow_tripdata_2009-01.csv.gz,yellow_tripdata_2009-01.csv.gz,504262564,1596568279000
dbfs:/databricks-datasets/nyctaxi/tripdata/yellow/yellow_tripdata_2009-02.csv.gz,yellow_tripdata_2009-02.csv.gz,480034681,1596568279000
dbfs:/databricks-datasets/nyctaxi/tripdata/yellow/yellow_tripdata_2009-03.csv.gz,yellow_tripdata_2009-03.csv.gz,521102719,1596568279000
dbfs:/databricks-datasets/nyctaxi/tripdata/yellow/yellow_tripdata_2009-04.csv.gz,yellow_tripdata_2009-04.csv.gz,515435466,1596568279000
dbfs:/databricks-datasets/nyctaxi/tripdata/yellow/yellow_tripdata_2009-05.csv.gz,yellow_tripdata_2009-05.csv.gz,531133739,1596568279000
dbfs:/databricks-datasets/nyctaxi/tripdata/yellow/yellow_tripdata_2009-06.csv.gz,yellow_tripdata_2009-06.csv.gz,508802995,1596568313000
dbfs:/databricks-datasets/nyctaxi/tripdata/yellow/yellow_tripdata_2009-07.csv.gz,yellow_tripdata_2009-07.csv.gz,487731497,1596568318000
dbfs:/databricks-datasets/nyctaxi/tripdata/yellow/yellow_tripdata_2009-08.csv.gz,yellow_tripdata_2009-08.csv.gz,490825210,1596568318000
dbfs:/databricks-datasets/nyctaxi/tripdata/yellow/yellow_tripdata_2009-09.csv.gz,yellow_tripdata_2009-09.csv.gz,503121179,1596568318000
dbfs:/databricks-datasets/nyctaxi/tripdata/yellow/yellow_tripdata_2009-10.csv.gz,yellow_tripdata_2009-10.csv.gz,567109604,1596568319000


**3 - Definição do schema**

Antes da criação do Structured Streaming, um arquivo da fonte será lido em modo batch para identificar o schema dos dados.

O schema identificado será posteriormente informado explicitamente ao `readStream`.

In [0]:
arquivo_exemplo = f"{path_fonte}yellow_tripdata_2019-01.csv.gz"

df_exemplo = (
    spark.read
        .option("header", True)
        .option("inferSchema", True)
        .csv(arquivo_exemplo)
)

df_exemplo.printSchema()

root
 |-- VendorID: integer (nullable = true)
 |-- tpep_pickup_datetime: timestamp (nullable = true)
 |-- tpep_dropoff_datetime: timestamp (nullable = true)
 |-- passenger_count: integer (nullable = true)
 |-- trip_distance: double (nullable = true)
 |-- RatecodeID: integer (nullable = true)
 |-- store_and_fwd_flag: string (nullable = true)
 |-- PULocationID: integer (nullable = true)
 |-- DOLocationID: integer (nullable = true)
 |-- payment_type: integer (nullable = true)
 |-- fare_amount: double (nullable = true)
 |-- extra: double (nullable = true)
 |-- mta_tax: double (nullable = true)
 |-- tip_amount: double (nullable = true)
 |-- tolls_amount: double (nullable = true)
 |-- improvement_surcharge: double (nullable = true)
 |-- total_amount: double (nullable = true)
 |-- congestion_surcharge: double (nullable = true)



In [0]:
schema_streaming = df_exemplo.schema

**4 - Configuração dos diretórios do streaming**

Serão utilizados diretórios específicos para controlar a entrada incremental dos arquivos e o checkpoint do Structured Streaming.

- **input:** receberá os arquivos que serão processados pelo streaming.
- **checkpoint:** armazenará informações necessárias para controle e recuperação do processamento.

In [0]:
streaming_base = "/Volumes/nyc_taxi_data/raw/landing/streaming"

streaming_input = f"{streaming_base}/input"
streaming_checkpoint = f"{streaming_base}/checkpoint"

dbutils.fs.mkdirs(streaming_input)
dbutils.fs.mkdirs(streaming_checkpoint)

True

In [0]:
display(dbutils.fs.ls(streaming_base))

path,name,size,modificationTime
dbfs:/Volumes/nyc_taxi_data/raw/landing/streaming/checkpoint/,checkpoint/,0,1787667139552
dbfs:/Volumes/nyc_taxi_data/raw/landing/streaming/input/,input/,0,1787667139552


**5 - Preparação da simulação incremental**

Para simular a chegada contínua de novos dados, serão criados pequenos lotes a partir do arquivo de exemplo.

Os lotes serão inicialmente armazenados em uma área de staging e posteriormente movidos para o diretório `input` em momentos diferentes, permitindo demonstrar o processamento incremental pelo Structured Streaming.

In [0]:
streaming_stage = f"{streaming_base}/stage"

dbutils.fs.mkdirs(streaming_stage)

True

In [0]:
lote_01 = (
    df_exemplo
    .filter(
        (col("tpep_pickup_datetime") >= "2019-01-01 00:00:00") &
        (col("tpep_pickup_datetime") < "2019-01-02 00:00:00")
    )
    .limit(1000)
)

lote_02 = (
    df_exemplo
    .filter(
        (col("tpep_pickup_datetime") >= "2019-01-02 00:00:00") &
        (col("tpep_pickup_datetime") < "2019-01-03 00:00:00")
    )
    .limit(1000)
)

lote_atrasado = (
    df_exemplo
    .filter(
        col("tpep_pickup_datetime") < "2019-01-01 00:00:00"
    )
    .limit(200)
)

In [0]:
print(f"Registros lote 01: {lote_01.count()}")
print(f"Registros lote 02: {lote_02.count()}")
print(f"Registros lote atrasado: {lote_atrasado.count()}")

Registros lote 01: 1000
Registros lote 02: 1000
Registros lote atrasado: 200


In [0]:
lote_01.write \
    .mode("overwrite") \
    .parquet(f"{streaming_stage}/lote_01")

lote_02.write \
    .mode("overwrite") \
    .parquet(f"{streaming_stage}/lote_02")

lote_atrasado.write \
    .mode("overwrite") \
    .parquet(f"{streaming_stage}/lote_atrasado")

In [0]:
display(dbutils.fs.ls(streaming_stage))

path,name,size,modificationTime
dbfs:/Volumes/nyc_taxi_data/raw/landing/streaming/stage/lote_01/,lote_01/,0,1787667258019
dbfs:/Volumes/nyc_taxi_data/raw/landing/streaming/stage/lote_02/,lote_02/,0,1787667258019
dbfs:/Volumes/nyc_taxi_data/raw/landing/streaming/stage/lote_atrasado/,lote_atrasado/,0,1787667258019


**6 - Criação do Structured Streaming**

O diretório input será utilizado como fonte incremental do Spark Structured Streaming. 

A cada execução com AvailableNow, novos arquivos adicionados ao diretório serão processados, enquanto o checkpoint impedirá o reprocessamento dos arquivos já consumidos.

In [0]:
df_stream = (
    spark.readStream
        .format("parquet")
        .schema(schema_streaming)
        .option("recursiveFileLookup", "true")
        .load(streaming_input)
)

In [0]:
print(f"DataFrame é streaming? {df_stream.isStreaming}")

DataFrame é streaming? True


In [0]:
df_stream.printSchema()

root
 |-- VendorID: integer (nullable = true)
 |-- tpep_pickup_datetime: timestamp (nullable = true)
 |-- tpep_dropoff_datetime: timestamp (nullable = true)
 |-- passenger_count: integer (nullable = true)
 |-- trip_distance: double (nullable = true)
 |-- RatecodeID: integer (nullable = true)
 |-- store_and_fwd_flag: string (nullable = true)
 |-- PULocationID: integer (nullable = true)
 |-- DOLocationID: integer (nullable = true)
 |-- payment_type: integer (nullable = true)
 |-- fare_amount: double (nullable = true)
 |-- extra: double (nullable = true)
 |-- mta_tax: double (nullable = true)
 |-- tip_amount: double (nullable = true)
 |-- tolls_amount: double (nullable = true)
 |-- improvement_surcharge: double (nullable = true)
 |-- total_amount: double (nullable = true)
 |-- congestion_surcharge: double (nullable = true)



**7 - Persistência do streaming em Delta Lake**

Devido à limitação do compute do Databricks Free Edition, será utilizado o trigger AvailableNow, que processa todos os dados disponíveis no momento da execução e encerra ao final do processamento.

In [0]:
tabela_streaming = "nyc_taxi_data.bronze.viagens_streaming"

In [0]:
query_streaming = (
    df_stream.writeStream
        .format("delta")
        .outputMode("append")
        .option("checkpointLocation", streaming_checkpoint)
        .trigger(availableNow=True)
        .toTable(tabela_streaming)
)

In [0]:
dbutils.fs.cp(
    f"{streaming_stage}/lote_01",
    f"{streaming_input}/lote_01",
    recurse=True
)

True

In [0]:
display(dbutils.fs.ls(streaming_input))

path,name,size,modificationTime
dbfs:/Volumes/nyc_taxi_data/raw/landing/streaming/input/lote_01/,lote_01/,0,1787667787144


In [0]:
query_streaming = (
    df_stream.writeStream
        .format("delta")
        .outputMode("append")
        .option("checkpointLocation", streaming_checkpoint)
        .trigger(availableNow=True)
        .toTable(tabela_streaming)
)

query_streaming.awaitTermination()

In [0]:
spark.sql("""
SELECT COUNT(*) AS Qtd_Registros
FROM nyc_taxi_data.bronze.viagens_streaming
""").show()

+-------------+
|Qtd_Registros|
+-------------+
|         1000|
+-------------+



**8 - Validação do processamento incremental**

Após o processamento do primeiro lote, um segundo conjunto de arquivos será adicionado ao diretório de entrada.

O objetivo é validar o comportamento incremental do Structured Streaming e o uso do checkpoint, verificando que apenas os novos arquivos sejam processados.

In [0]:
dbutils.fs.cp(
    f"{streaming_stage}/lote_02",
    f"{streaming_input}/lote_02",
    recurse=True
)

True

In [0]:
display(dbutils.fs.ls(streaming_input))

path,name,size,modificationTime
dbfs:/Volumes/nyc_taxi_data/raw/landing/streaming/input/lote_01/,lote_01/,0,1787667862582
dbfs:/Volumes/nyc_taxi_data/raw/landing/streaming/input/lote_02/,lote_02/,0,1787667862582


In [0]:
query_streaming = (
    df_stream.writeStream
        .format("delta")
        .outputMode("append")
        .option("checkpointLocation", streaming_checkpoint)
        .trigger(availableNow=True)
        .toTable(tabela_streaming)
)

query_streaming.awaitTermination()

In [0]:
spark.sql("""
SELECT COUNT(*) AS Qtd_Registros
FROM nyc_taxi_data.bronze.viagens_streaming
""").show()

+-------------+
|Qtd_Registros|
+-------------+
|         2000|
+-------------+



**Resultado da validação incremental**

Após a primeira execução do Structured Streaming, a tabela Delta continha 1.000 registros referentes ao primeiro lote.

Em seguida, um segundo lote contendo mais 1.000 registros foi adicionado ao diretório de entrada. Uma nova execução do streaming utilizando o mesmo diretório de checkpoint resultou em um total de 2.000 registros na tabela.

Como o primeiro lote permaneceu no diretório de entrada e não foi processado novamente, o teste demonstra o funcionamento do processamento incremental e do mecanismo de checkpoint do Spark Structured Streaming.

In [0]:
dbutils.fs.cp(
    f"{streaming_stage}/lote_atrasado",
    f"{streaming_input}/lote_atrasado",
    recurse=True
)

True

In [0]:
display(dbutils.fs.ls(streaming_input))

path,name,size,modificationTime
dbfs:/Volumes/nyc_taxi_data/raw/landing/streaming/input/lote_01/,lote_01/,0,1787667983981
dbfs:/Volumes/nyc_taxi_data/raw/landing/streaming/input/lote_02/,lote_02/,0,1787667983981
dbfs:/Volumes/nyc_taxi_data/raw/landing/streaming/input/lote_atrasado/,lote_atrasado/,0,1787667983981


In [0]:
query_streaming = (
    df_stream.writeStream
        .format("delta")
        .outputMode("append")
        .option("checkpointLocation", streaming_checkpoint)
        .trigger(availableNow=True)
        .toTable(tabela_streaming)
)

query_streaming.awaitTermination()

In [0]:
spark.sql("""
SELECT COUNT(*) AS Qtd_Registros
FROM nyc_taxi_data.bronze.viagens_streaming
""").show()

+-------------+
|Qtd_Registros|
+-------------+
|         2200|
+-------------+



### Resultado do processamento incremental

Foram realizadas três execuções do Structured Streaming utilizando o mesmo diretório de checkpoint:

- Primeiro lote: 1.000 registros processados.
- Segundo lote: mais 1.000 registros, totalizando 2.000.
- Lote com eventos atrasados: mais 200 registros, totalizando 2.200.

Os arquivos processados anteriormente permaneceram no diretório de entrada, mas não foram ingeridos novamente, demonstrando o controle incremental realizado pelo mecanismo de checkpoint do Spark Structured Streaming.

**9 - Watermark e agregação temporal**

Para demonstrar o tratamento de eventos atrasados, será utilizada a coluna `tpep_pickup_datetime` como tempo do evento.

O Structured Streaming aplicará watermark e agregações em janelas temporais, permitindo limitar o estado mantido pelo Spark e controlar o tratamento de registros que chegam com atraso.

In [0]:
df_stream_watermark = (
    df_stream
        .withWatermark("tpep_pickup_datetime", "1 day")
        .groupBy(
            window(
                col("tpep_pickup_datetime"),
                "1 hour"
            )
        )
        .agg(
            count("*").alias("Qtd_Viagens"),
            avg("trip_distance").alias("Distancia_Media"),
            avg("total_amount").alias("Valor_Medio_Corrida")
        )
)

In [0]:
print(f"DataFrame é streaming? {df_stream_watermark.isStreaming}")

DataFrame é streaming? True


In [0]:
df_stream_watermark.printSchema()

root
 |-- window: struct (nullable = false)
 |    |-- start: timestamp (nullable = true)
 |    |-- end: timestamp (nullable = true)
 |-- Qtd_Viagens: long (nullable = false)
 |-- Distancia_Media: double (nullable = true)
 |-- Valor_Medio_Corrida: double (nullable = true)



In [0]:
watermark_base = f"{streaming_base}/watermark"

watermark_input = f"{watermark_base}/input"
watermark_checkpoint = f"{watermark_base}/checkpoint"

dbutils.fs.mkdirs(watermark_input)
dbutils.fs.mkdirs(watermark_checkpoint)

display(dbutils.fs.ls(watermark_base))

path,name,size,modificationTime
dbfs:/Volumes/nyc_taxi_data/raw/landing/streaming/watermark/checkpoint/,checkpoint/,0,1787668245553
dbfs:/Volumes/nyc_taxi_data/raw/landing/streaming/watermark/input/,input/,0,1787668245553


In [0]:
df_stream_watermark_base = (
    spark.readStream
        .format("parquet")
        .schema(schema_streaming)
        .option("recursiveFileLookup", "true")
        .load(watermark_input)
)

In [0]:
df_stream_watermark = (
    df_stream_watermark_base
        .withWatermark("tpep_pickup_datetime", "1 day")
        .groupBy(
            window(
                col("tpep_pickup_datetime"),
                "1 hour"
            )
        )
        .agg(
            count("*").alias("Qtd_Viagens"),
            avg("trip_distance").alias("Distancia_Media"),
            avg("total_amount").alias("Valor_Medio_Corrida")
        )
)

In [0]:
print(f"DataFrame base é streaming? {df_stream_watermark_base.isStreaming}")
print(f"DataFrame agregado é streaming? {df_stream_watermark.isStreaming}")

df_stream_watermark.printSchema()

DataFrame base é streaming? True
DataFrame agregado é streaming? True
root
 |-- window: struct (nullable = false)
 |    |-- start: timestamp (nullable = true)
 |    |-- end: timestamp (nullable = true)
 |-- Qtd_Viagens: long (nullable = false)
 |-- Distancia_Media: double (nullable = true)
 |-- Valor_Medio_Corrida: double (nullable = true)



In [0]:
dbutils.fs.cp(
    f"{streaming_stage}/lote_01",
    f"{watermark_input}/lote_01",
    recurse=True
)

True

In [0]:
display(dbutils.fs.ls(watermark_input))

path,name,size,modificationTime
dbfs:/Volumes/nyc_taxi_data/raw/landing/streaming/watermark/input/lote_01/,lote_01/,0,1787668402802


In [0]:
tabela_watermark = "nyc_taxi_data.gold.viagens_streaming_watermark"

query_watermark = (
    df_stream_watermark.writeStream
        .format("delta")
        .outputMode("append")
        .option("checkpointLocation", watermark_checkpoint)
        .trigger(availableNow=True)
        .toTable(tabela_watermark)
)

query_watermark.awaitTermination()

In [0]:
spark.table(tabela_watermark).show(20, truncate=False)

+------+-----------+---------------+-------------------+
|window|Qtd_Viagens|Distancia_Media|Valor_Medio_Corrida|
+------+-----------+---------------+-------------------+
+------+-----------+---------------+-------------------+



In [0]:
print(query_watermark.lastProgress)

{
    "id": "580865a0-fd94-471e-a067-1f887364399d",
    "runId": "9adf0e10-a35c-41f9-b58d-61a36fcfd15a",
    "name": null,
    "timestamp": "2026-08-25T14:33:52.910Z",
    "batchId": 1,
    "batchDuration": 4746,
    "durationMs": {
        "triggerExecution": 4746,
        "queryPlanning": 199,
        "collectSourceMetrics": 0,
        "getBatch": 2,
        "commitOffsets": 80,
        "addBatch": 4216,
        "latestOffset": 1,
        "commitBatch": 143,
        "walCommit": 180
    },
    "eventTime": {
        "watermark": "2018-12-31T01:03:03.000Z"
    },
    "stateOperators": [
        {
            "operatorName": "stateStoreSave",
            "numRowsTotal": 2,
            "numRowsUpdated": 0,
            "allUpdatesTimeMs": 231,
            "numRowsRemoved": 0,
            "allRemovalsTimeMs": 3246,
            "commitTimeMs": 10472,
            "memoryUsedBytes": 269064422,
            "numRowsDroppedByWatermark": 0,
            "numShufflePartitions": 200,
            "n

In [0]:
dbutils.fs.cp(
    f"{streaming_stage}/lote_02",
    f"{watermark_input}/lote_02",
    recurse=True
)

True

In [0]:
display(dbutils.fs.ls(watermark_input))

path,name,size,modificationTime
dbfs:/Volumes/nyc_taxi_data/raw/landing/streaming/watermark/input/lote_01/,lote_01/,0,1787668704979
dbfs:/Volumes/nyc_taxi_data/raw/landing/streaming/watermark/input/lote_02/,lote_02/,0,1787668704979


In [0]:
query_watermark = (
    df_stream_watermark.writeStream
        .format("delta")
        .outputMode("append")
        .option("checkpointLocation", watermark_checkpoint)
        .trigger(availableNow=True)
        .toTable(tabela_watermark)
)

query_watermark.awaitTermination()

In [0]:
spark.table(tabela_watermark).show(50, truncate=False)

+------+-----------+---------------+-------------------+
|window|Qtd_Viagens|Distancia_Media|Valor_Medio_Corrida|
+------+-----------+---------------+-------------------+
+------+-----------+---------------+-------------------+



In [0]:
print(query_watermark.lastProgress)

{
    "id": "580865a0-fd94-471e-a067-1f887364399d",
    "runId": "cca1f772-94b3-4fca-bf98-81cf348db7c7",
    "name": null,
    "timestamp": "2026-08-25T14:38:54.170Z",
    "batchId": 3,
    "batchDuration": 4091,
    "durationMs": {
        "triggerExecution": 4091,
        "queryPlanning": 193,
        "collectSourceMetrics": 0,
        "getBatch": 0,
        "commitOffsets": 72,
        "addBatch": 3566,
        "latestOffset": 0,
        "commitBatch": 181,
        "walCommit": 148
    },
    "eventTime": {
        "watermark": "2019-01-01T00:59:54.000Z"
    },
    "stateOperators": [
        {
            "operatorName": "stateStoreSave",
            "numRowsTotal": 3,
            "numRowsUpdated": 0,
            "allUpdatesTimeMs": 62,
            "numRowsRemoved": 0,
            "allRemovalsTimeMs": 2250,
            "commitTimeMs": 9056,
            "memoryUsedBytes": 269071133,
            "numRowsDroppedByWatermark": 0,
            "numShufflePartitions": 200,
            "num

In [0]:
dbutils.fs.cp(
    f"{streaming_stage}/lote_atrasado",
    f"{watermark_input}/lote_atrasado",
    recurse=True
)

True

In [0]:
query_watermark = (
    df_stream_watermark.writeStream
        .format("delta")
        .outputMode("append")
        .option("checkpointLocation", watermark_checkpoint)
        .trigger(availableNow=True)
        .toTable(tabela_watermark)
)

query_watermark.awaitTermination()

In [0]:
for progresso in query_watermark.recentProgress:
    print("Batch:", progresso["batchId"])
    print("Input rows:", progresso["numInputRows"])
    print("Watermark:", progresso.get("eventTime", {}).get("watermark"))
    
    if len(progresso.get("stateOperators", [])) > 0:
        print(
            "Dropped by watermark:",
            progresso["stateOperators"][0]["numRowsDroppedByWatermark"]
        )
    
    print("-" * 50)

Batch: 4
Input rows: None
Watermark: 2019-01-01T00:59:54.000Z
Dropped by watermark: 17
--------------------------------------------------


### Resultado do teste de Watermark

Após o processamento dos lotes iniciais, o watermark avançou para:

`2019-01-01 00:59:54`

Em seguida, foi inserido um lote contendo eventos com timestamps anteriores ao período já processado.

O Spark Structured Streaming identificou registros atrasados em relação ao watermark e descartou 17 registros da agregação temporal.

Esse comportamento demonstra o uso do watermark para controle de estado e tratamento de dados atrasados em operações stateful.

O lote atrasado tinha 200 registros, mas apenas 17 foram descartados pelo watermark. Isso é normal: o Spark não descarta automaticamente todo dado “antigo”; ele avalia cada evento em relação ao watermark e à janela temporal correspondente.

In [0]:
spark.table(tabela_watermark).orderBy("window").show(50, truncate=False)

+------+-----------+---------------+-------------------+
|window|Qtd_Viagens|Distancia_Media|Valor_Medio_Corrida|
+------+-----------+---------------+-------------------+
+------+-----------+---------------+-------------------+



### Conclusão da etapa de Streaming

O pipeline implementado demonstrou:

- leitura incremental com `readStream`;
- escrita incremental em Delta Lake com `writeStream`;
- uso de `checkpoint`;
- processamento com trigger `AvailableNow`;
- ingestão de múltiplos lotes sem reprocessamento;
- uso de event time;
- agregação temporal em janelas de 1 hora;
- watermark de 1 dia;
- tratamento de dados atrasados;
- descarte de registros além do watermark.